# Physics Informed Deep Learning (Part II): Data-driven Discovery of Nonlinear Partial Differential Equations

**Paper:** Raissi, M., Perdikaris, P., Karniadakis, G.E. (2017). *Physics Informed Deep Learning (Part II): Data-driven Discovery of Nonlinear Partial Differential Equations.* arXiv:1711.10566.

**Carpeta origen:** `PINNs/4. Otros/Physics Informed Deep Learning (Part II) Data-driven.pdf`

## Como se usan las PINNs en este paper

Mientras la Parte I (ya cubierta en esta coleccion, carpetas "1. mecanica de fluidos" y "4. Otros") resuelve una EDP con parametros $\lambda$ **conocidos**, esta Parte II aborda el problema inverso/de **descubrimiento**: dado un conjunto **disperso y ruidoso** de observaciones de $u(t,x)$, encontrar los parametros $\lambda$ que mejor explican los datos. Para la ecuacion de Burgers (Eq. 3):

$$u_t+\lambda_1 uu_x-\lambda_2 u_{xx}=0$$

se define $f:=u_t+\lambda_1uu_x-\lambda_2u_{xx}$ (Eq. 4) igual que en la Parte I, pero ahora **$\lambda_1,\lambda_2$ son tambien parametros entrenables** de la red (ademas de los pesos y sesgos), optimizados conjuntamente minimizando (Eq. 5):

$$MSE=MSE_u+MSE_f,\qquad MSE_u=\frac{1}{N}\sum_i|u(t_u^i,x_u^i)-u^i|^2,\qquad MSE_f=\frac{1}{N}\sum_i|f(t_u^i,x_u^i)|^2$$

donde ahora $MSE_u$ compara contra **datos dispersos observados de $u$** (no condiciones de contorno) y $MSE_f$ fuerza que la EDP se satisfaga en esos mismos puntos con los $\lambda$ actuales. El paper entrena con $N=2000$ puntos aleatorios de todo el dominio espacio-temporal ($\lambda_1=1.0$, $\lambda_2=0.01/\pi$ verdaderos), y muestra (Fig. 1, Tabla 1) que el metodo recupera $\lambda_1,\lambda_2$ con gran precision, incluso con **10% de ruido** en los datos.

Este cuaderno reproduce fielmente este **ejemplo de descubrimiento con la ecuacion de Burgers** (Seccion 2.1): se genera una solucion de referencia numerica de alta resolucion (diferencias finitas + integrador stiff), se muestrean $N=2000$ puntos dispersos con ruido, y se entrena una PINN que recupera simultaneamente $u(t,x)$ y los parametros $(\lambda_1,\lambda_2)$, reportando el error porcentual igual que en la Tabla 1 del paper.

## Repositorio publico

El paper **incluye explicitamente** el enlace a su repositorio oficial en el propio texto (Seccion 1): "All data and codes used in this manuscript are publicly available on GitHub at https://github.com/maziarraissi/PINNs".

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs (carpeta `appendix/continuous_time_identification (Burgers)/`)

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib scipy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Generacion de la solucion de referencia (Eq. 3, $\lambda_1=1$, $\lambda_2=0.01/\pi$) via diferencias finitas

In [ ]:
lambda1_true, lambda2_true = 1.0, 0.01 / np.pi

N_x = 401
x_grid = np.linspace(-1, 1, N_x)
dx = x_grid[1] - x_grid[0]
u0 = -np.sin(np.pi * x_grid[1:-1])  # u(0,x) = -sin(pi x), interior (BC u=0 en x=-1,1)

def burgers_rhs(t, u_int, dx, nu):
    u_full = np.concatenate([[0.0], u_int, [0.0]])
    u_x = (u_full[2:] - u_full[:-2]) / (2 * dx)
    u_xx = (u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]) / dx**2
    return -u_full[1:-1] * u_x + nu * u_xx

t_eval = np.linspace(0, 1, 100)
sol = solve_ivp(burgers_rhs, [0, 1], u0, method='Radau', t_eval=t_eval,
                 args=(dx, lambda2_true), rtol=1e-8, atol=1e-10)
U_ref = np.zeros((len(t_eval), N_x))
U_ref[:, 1:-1] = sol.y.T
print(f'Referencia generada: {U_ref.shape[0]} instantes x {U_ref.shape[1]} puntos espaciales')

plt.figure(figsize=(7, 4))
plt.pcolormesh(t_eval, x_grid, U_ref.T, cmap='RdBu_r', shading='auto')
plt.xlabel('t'); plt.ylabel('x'); plt.title('u(t,x) de referencia (Burgers, diferencias finitas)')
plt.colorbar()
plt.show()

## 2. Muestreo disperso con ruido (N=2000, como en el paper) y red PINN con $\lambda_1,\lambda_2$ entrenables

In [ ]:
N_data = 2000
noise_level = 0.01  # 1% de ruido, el caso principal del paper (Fig. 1)

it = np.random.randint(0, len(t_eval), N_data)
ix = np.random.randint(0, N_x, N_data)
t_data = t_eval[it]
x_data = x_grid[ix]
u_data = U_ref[it, ix]
u_data_noisy = u_data + noise_level * np.std(u_data) * np.random.randn(N_data)

tx_data = torch.tensor(np.stack([t_data, x_data], axis=1), dtype=torch.float32, device=device)
u_data_t = torch.tensor(u_data_noisy, dtype=torch.float32, device=device).view(-1, 1)


class BurgersDiscoveryPINN(nn.Module):
    def __init__(self, n_hidden=8, n_neurons=20):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)
        # lambda_1, lambda_2 son PARAMETROS ENTRENABLES junto con la red (Eq. 4-5)
        self.lambda1 = nn.Parameter(torch.tensor(0.0))
        self.lambda2 = nn.Parameter(torch.tensor(-6.0))  # log-param, ver forward

    def forward(self, tx):
        return self.net(tx)


model = BurgersDiscoveryPINN().to(device)


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]

## 3. Perdida (Eq. 4-5): $MSE_u$ (ajuste a los datos dispersos) + $MSE_f$ (residuo con los $\lambda$ actuales)

In [ ]:
tx_col = tx_data.clone().requires_grad_(True)  # mismos puntos para datos y colocacion, como en el paper

def compute_loss(model):
    u_pred = model(tx_col)
    loss_u = torch.mean((u_pred - u_data_t)**2)

    u_t = d_d(u_pred, tx_col, 0)
    u_x = d_d(u_pred, tx_col, 1)
    u_xx = d_d(u_x, tx_col, 1)

    lambda2_pos = torch.exp(model.lambda2)  # garantiza lambda2 > 0 (difusividad fisica)
    f = u_t + model.lambda1 * u_pred * u_x - lambda2_pos * u_xx
    loss_f = torch.mean(f**2)

    return loss_u + loss_f, model.lambda1.item(), lambda2_pos.item()

## 4. Entrenamiento (el paper usa L-BFGS)

In [ ]:
opt_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(4000):
    opt_adam.zero_grad()
    loss, l1, l2 = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if epoch % 1000 == 0:
        print(f'[Adam] epoch {epoch:5d} | loss={loss.item():.4e} | lambda1={l1:.4f} | lambda2={l2:.6f}')

opt_lbfgs = torch.optim.LBFGS(model.parameters(), lr=0.5, max_iter=500,
                               history_size=50, line_search_fn='strong_wolfe')

def closure():
    opt_lbfgs.zero_grad()
    loss, _, _ = compute_loss(model)
    loss.backward()
    return loss

opt_lbfgs.step(closure)
_, l1_final, l2_final = compute_loss(model)
print(f'[L-BFGS] lambda1 final={l1_final:.5f} | lambda2 final={l2_final:.6f}')

## 5. Resultados: error porcentual en los parametros identificados (cf. Tabla 1 del paper)

In [ ]:
err_l1 = 100 * abs(l1_final - lambda1_true) / lambda1_true
err_l2 = 100 * abs(l2_final - lambda2_true) / lambda2_true

print(f'lambda1: verdadero={lambda1_true:.5f}, identificado={l1_final:.5f}, error={err_l1:.3f}%')
print(f'lambda2: verdadero={lambda2_true:.6f}, identificado={l2_final:.6f}, error={err_l2:.3f}%')
print(f'(paper reporta, con ruido 1%: {0.518}% y {0.483}% de error tipicamente, Tabla 1)')

with torch.no_grad():
    tx_grid = torch.tensor(np.stack([np.repeat(t_eval, N_x), np.tile(x_grid, len(t_eval))], axis=1),
                            dtype=torch.float32, device=device)
    u_pred_grid = model(tx_grid).cpu().numpy().reshape(len(t_eval), N_x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
im0 = axes[0].pcolormesh(t_eval, x_grid, U_ref.T, cmap='RdBu_r', shading='auto')
axes[0].set_title('u(t,x) de referencia'); axes[0].set_xlabel('t'); axes[0].set_ylabel('x')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].pcolormesh(t_eval, x_grid, u_pred_grid.T, cmap='RdBu_r', shading='auto')
axes[1].set_title('u(t,x) PINN (con lambda recuperados)'); axes[1].set_xlabel('t')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()